In [1]:
# Some basics
import math
import os
import pandas as pd
import numpy as np
import time
from tqdm import tqdm

# Declare filenames
ROOT_DATA_DIR = (r"~\Data\Eyetracking_03_Matched_frames") # folder where the Input data sits
DATA_DIR_OUTPUT =  (r"~\Data\Eyetracking_04_Match_Segmentation")
SEMANTIC_SEGMENTATION_DIR = (r'~\Data\Semantic_01_Segmentation\Robust_batch')

# Define the column names for each type of eyetracking data
sample_columns   = ['tSample', 'LX', 'LY', 'LPupil', 'RX', 'RY', 'RPupil', 'Movie', 'matched_frame_IDx']

# Define a dictionary to map movie labels to their corresponding video files
movie_files = {
    "movie_01": "Charite",
    "movie_02": "Ziemlich_Beste_Freunde",
    "movie_03": "High_Seas",
    "movie_04": "Biohackers",
    "movie_05": "Downton_Abbey",
    "movie_06": "New_Amsterdam"
}

# Define a mapping of eyetracking types to coordinates
coordinates_mapping = { "Samples": ['LX', 'LY', 'RX', 'RY']}

frame_width = 1921
frame_height = 1081


# Iterate through each eye tracking file
for participant_folder in os.listdir(ROOT_DATA_DIR):
    
    # Construct the full path to the participant subfolder
    participant_folder_path = os.path.join(ROOT_DATA_DIR, participant_folder)
    #print('Participant Directory:', participant_folder_path)

    # Check if the subfolder is a directory
    if os.path.isdir(participant_folder_path):

        # Filter the eyetracking files based on the presence of "Samples" in their filenames
        sample_files = [eyetracking_file for eyetracking_file in os.listdir(participant_folder_path) if 'Samples' in eyetracking_file]
        
        for eyetracking_file in sample_files:  # Assuming your eye tracking files have a .csv extension
                      
            # Construct the output file path
            output_file = os.path.join(DATA_DIR_OUTPUT, f'{eyetracking_file[:-19]}_Pixel_Category.csv')
            output_file_path = os.path.join(DATA_DIR_OUTPUT, output_file)
            #print(f'OutputFile', output_file)

            # Check if the output file already exists
            if os.path.exists(output_file_path):
                #print(f"Output file already exists for {eyetracking_file}. Skipping to the next file.")
                continue
            
            file_path = os.path.join(participant_folder_path, eyetracking_file)
            movie_name = None

            # Extract the information from the eyetracking file name
            file_parts = eyetracking_file[:-3].split("_")
            participant = file_parts[0] 
            block_number = file_parts[-7]
            order_number = file_parts[-5]
            eyetracking_type = file_parts[-9]

            # Load the eye tracking data
            start_time_eyetracking_data = time.time()
            eyetracking_data = pd.read_csv((file_path), delimiter=',')

            end_time_eyetracking_data = time.time()
            load_time = end_time_eyetracking_data - start_time_eyetracking_data
            #print(f"Eye tracking data loaded in {load_time} seconds.")

            eyetracking_data = eyetracking_data[sample_columns]
            eyetracking_data[['LX', 'LY', 'RX', 'RY']] = eyetracking_data[['LX', 'LY', 'RX', 'RY']].apply(np.floor)

            # Extract the desired columns from the eyetracking data if they exist in the file
            eyetracking_data = eyetracking_data[sample_columns]
            movie_name = eyetracking_data['Movie'].loc[0]
            #print(movie_name)
        
            eyetracking_data['MatchingPixel'] = -1

            prev_frame_data = None
            prev_frame_index = -1

            eyetracking_data['OutOfBounds'] = ((eyetracking_data['LX'] < 0) | (eyetracking_data['LX'] >= frame_width) | 
                                   (eyetracking_data['LY'] < 0) | (eyetracking_data['LY'] >= frame_height) |
                                   (eyetracking_data['RX'] < 0) | (eyetracking_data['RX'] >= frame_width) |
                                   (eyetracking_data['RY'] < 0) | (eyetracking_data['RY'] >= frame_height))

            for mask_folder in os.listdir(SEMANTIC_SEGMENTATION_DIR):
                if mask_folder.startswith(movie_files[movie_name][0]):
                    mask_folder_path = os.path.join(SEMANTIC_SEGMENTATION_DIR, mask_folder)
                    total_frames = len(os.listdir(mask_folder_path))

                    # Check if there are empty MatchingPixel values
                    if eyetracking_data['MatchingPixel'].isnull().any():
                        # Find the first index with an empty MatchingPixel value
                        first_empty_index = eyetracking_data.loc[eyetracking_data['MatchingPixel'].isnull()].index[0]
                        # Start from the next index
                        start_index = first_empty_index + 1
                    else:
                        start_index = 0
                    

                    for index, row in tqdm(eyetracking_data.iterrows(), total=eyetracking_data.shape[0]):

                        if row['OutOfBounds']:
                            continue  # Skip this row if the coordinate is out-of-bounds

                        frame_index_start = row['matched_frame_IDx']

                        # Skip this row if the frame index is out-of-bounds (total_frames/2 because the folder contains both csv files and images)
                        if frame_index_start < 0 or frame_index_start >= (total_frames/2):
                            continue

                        LX = row['LX']
                        LY = row['LY']
                        RX = row['RX']
                        RY = row['RY']

                        
                        # If this frame_index is different from the previous one, load the new frame
                        if frame_index_start != prev_frame_index:
                            # Load the frame data
                            frame_filename = f"{mask_folder}_frame_{frame_index_start:04}.csv"
                            frame_file_path = os.path.join(mask_folder_path, frame_filename)
                            frame_data = pd.read_csv(frame_file_path, delimiter=';', header=None)
                            
                            
                            # Update the cached frame data and index
                            prev_frame_data = frame_data
                            prev_frame_index = frame_index_start
                        else:
                            # Use the cached frame data
                            frame_data = prev_frame_data


                        if not math.isnan(LX) and not math.isnan(LY):
                            LX_index = int(LX)
                            LY_index = int(LY)
                            # DataFrame.iloc[row_index, column_index]
                            matching_pixel = frame_data.iloc[LY_index, LX_index].astype(int)
                            eyetracking_data.at[index, 'MatchingPixel'] = matching_pixel

                        elif not math.isnan(RX) and not math.isnan(RY):
                            RX_index = int(RX)
                            RY_index = int(RY)
                            # DataFrame.iloc[row_index, column_index]
                            matching_pixel = frame_data.iloc[RY_index, RX_index].astype(int)
                            eyetracking_data.at[index, 'MatchingPixel'] = matching_pixel

                        #last_successful_frame = frame_index_start

            eyetracking_data.to_csv(output_file_path)
            print("Participant ", participant, "Block", block_number, "Order", order_number)
            print('File Saved')
            print("------------------------------")


  6%|▌         | 3866/65513 [00:45<22:45, 45.15it/s] 